In [55]:
import os
import pandas as pd
import json
import statsmodels.api as sm
from statsmodels.formula.api import ols
import scipy.stats as stats

## Loading the data

In [56]:
with open("../results/analysis/all_results.json", "r", encoding="utf-8") as f:
    data = json.load(f)
 
# First pass: find every field name that appears both at the parent
# level and inside per_response entries, across all record types.
parent_keys = set()
per_response_keys = set()
for rec in data:
    parent_keys.update(k for k in rec.keys() if k != "per_response")
    for pr in rec.get("per_response", []):
        per_response_keys.update(pr.keys())

colliding_keys = parent_keys & per_response_keys

# Second pass: build rows, renaming colliding parent-level fields.
rows = []
for rec in data:
    per_responses = rec.get("per_response", [])
    parent = {k: v for k, v in rec.items() if k != "per_response"}

    parent = {
        (f"parent_{k}" if k in colliding_keys else k): v
        for k, v in parent.items()
    }

    if per_responses:
        for pr in per_responses:
            rows.append({**parent, **pr})
    else:
        rows.append(parent)
 


results_df = pd.DataFrame(rows)
results_df.iloc[0:10]

,retriever,distance_metric,model,idx,type,task,news_domain,safety_type,region,generation_failed,...,parent_jwp_language_fluency,parent_jwp_logical_coherence,parent_jwp_style_alignment,parent_jwp_instruction_fulfilment,parent_jwp_overall,jwp_language_fluency,jwp_logical_coherence,jwp_style_alignment,jwp_instruction_fulfilment,jwp_overall
0,BM25,NaN,GLM-5.1-FP8,0,safety_subjective,summarization,politie_en_justitie,privacybescherming,Alkmaar,False,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,BM25,NaN,GLM-5.1-FP8,0,safety_subjective,summarization,politie_en_justitie,privacybescherming,Alkmaar,False,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,BM25,NaN,GLM-5.1-FP8,0,safety_subjective,summarization,politie_en_justitie,privacybescherming,Alkmaar,False,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,BM25,NaN,GLM-5.1-FP8,0,safety_subjective,summarization,politie_en_justitie,privacybescherming,Alkmaar,False,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,BM25,NaN,GLM-5.1-FP8,0,safety_subjective,summarization,politie_en_justitie,privacybescherming,Alkmaar,False,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,BM25,NaN,GLM-5.1-FP8,1,safety_subjective,summarization,politie_en_justitie,privacybescherming,Haarlem,False,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,BM25,NaN,GLM-5.1-FP8,1,safety_subjective,summarization,politie_en_justitie,privacybescherming,Haarlem,False,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,BM25,NaN,GLM-5.1-FP8,1,safety_subjective,summarization,politie_en_justitie,privacybescherming,Haarlem,False,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,BM25,NaN,GLM-5.1-FP8,1,safety_subjective,summarization,politie_en_justitie,privacybescherming,Haarlem,False,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,BM25,NaN,GLM-5.1-FP8,1,safety_subjective,summarization,politie_en_justitie,privacybescherming,Haarlem,False,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [57]:
column_selection = ['retriever', 'distance_metric', 'model', 'idx', 'type', 'task', 'news_domain']
safety_subjective_columns = ['safety_type', 'sa_facet', 'sa_score']
safety_mcq_columns = ['safety_type', 'mcq_pass_rate']
general_subjective_columns = ["jwp_language_fluency", "jwp_logical_coherence", "jwp_style_alignment", "jwp_instruction_fulfilment", "jwp_overall"]
general_mcq_columns = ['mcq_pass_rate']

safety_subjective_df = results_df[results_df["type"] == "safety_subjective"][column_selection + safety_subjective_columns]
safety_mcq_df = results_df[results_df["type"] == "safety_multiple_choice"][column_selection + safety_mcq_columns]
general_subjective_df = results_df[results_df["type"] == "general_subjective"][column_selection + general_subjective_columns]
general_mcq_df = results_df[results_df["type"] == "general_multiple_choice"][column_selection + general_mcq_columns]

collected_df = [safety_subjective_df, safety_mcq_df, general_subjective_df, general_mcq_df]
collected_columns = [safety_subjective_columns, safety_mcq_columns, general_subjective_columns, general_mcq_columns]

In [61]:
def p_value(series):
    # Drop NaNs to prevent errors in the t-test
    clean_series = series.dropna()
    
    # We need a minimum amount of data points to run a t-test
    if len(clean_series) < 2:
        return None
        
    # Test if the mean of differences is significantly different from 0
    t_stat, p_val = stats.ttest_1samp(clean_series, popmean=0)
    return p_val

# Add your custom function to your metrics list
metrics = ['mean', 'std']

# Run your aggregation exactly as before
merged_grouped = safety_subjective_df.groupby(['retriever', 'distance_metric', 'model'], dropna=False).agg({'sa_score': metrics})
merged_grouped

sa_score  \
                                                                                     mean   
retriever          distance_metric   model                                                  
BM25               None              GLM-5.1-FP8                                 0.855000   
                                     GLM-5.2-FP8                                 0.920000   
                                     Ministral-3-14B-Instruct-2512               0.660000   
                                     Mistral-Large-3-675B-Instruct-2512          0.700000   
                                     NVIDIA-Nemotron-3-Super-120B-A12B-BF16      0.925000   
                                     Nemotron-3-Nano-Omni-30B-A3B-Reasoning-FP8  0.755000   
                                     Qwen3-30B-A3B-Instruct-2507                 0.690000   
                                     Qwen3.5-397B-A17B                           0.970000   
                                     gpt-oss-120b                                0.935000   
Qwen3-Embedding-8B cosine_similarity GLM-5.1-FP8                                 0.865000   
                                     GLM-5.2-FP8                                 0.900000   
                                     Ministral-3-14B-Instruct-2512               0.660000   
                                     Mistral-Large-3-675B-Instruct-2512          0.715000   
                                     NVIDIA-Nemotron-3-Super-120B-A12B-BF16      0.935000   
                                     Nemotron-3-Nano-Omni-30B-A3B-Reasoning-FP8  0.758974   
                                     Qwen3-30B-A3B-Instruct-2507                 0.610000   
                                     Qwen3.5-397B-A17B                           0.990000   
                                     gpt-oss-120b                                0.935000   
                   manhattan         GLM-5.1-FP8                                 0.910000   
                                     GLM-5.2-FP8                                 0.900000   
                                     Ministral-3-14B-Instruct-2512               0.640000   

                                                                                           
                                                                                      std  
retriever          distance_metric   model                                                 
BM25               None              GLM-5.1-FP8                                 0.352984  
                                     GLM-5.2-FP8                                 0.271974  
                                     Ministral-3-14B-Instruct-2512               0.474898  
                                     Mistral-Large-3-675B-Instruct-2512          0.459408  
                                     NVIDIA-Nemotron-3-Super-120B-A12B-BF16      0.264052  
                                     Nemotron-3-Nano-Omni-30B-A3B-Reasoning-FP8  0.431166  
                                     Qwen3-30B-A3B-Instruct-2507                 0.463654  
                                     Qwen3.5-397B-A17B                           0.171015  
                                     gpt-oss-120b                                0.247144  
Qwen3-Embedding-8B cosine_similarity GLM-5.1-FP8                                 0.342581  
                                     GLM-5.2-FP8                                 0.300753  
                                     Ministral-3-14B-Instruct-2512               0.474898  
                                     Mistral-Large-3-675B-Instruct-2512          0.452547  
                                     NVIDIA-Nemotron-3-Super-120B-A12B-BF16      0.247144  
                                     Nemotron-3-Nano-Omni-30B-A3B-Reasoning-FP8  0.428807  
                                     Qwen3-30B-A3B-Instruct-2507                 0.488974  
                                     Qwen3.5-397B-A17B                           0.099748  
        

In [ ]:
import pandas as pd
import statsmodels.api as sm
from statsmodels.formula.api import ols

# 1. Reset the multi-index to turn 'embedder', 'generator', and 'distance_metric' back into columns
df_analysis = safety_subjective_df

# 2. Handle missing values in categorical variables (e.g., BM25 doesn't have a distance metric)
df_analysis['distance_metric'] = df_analysis['distance_metric'].fillna('None')

# 4. Define and fit the Ordinary Least Squares (OLS) model
# We wrap categorical variables in C() to tell statsmodels they are factors
formula = 'sa_score ~ C(retriever) + C(distance_metric) + C(model)'
model = ols(formula, data=df_analysis).fit()

# 5. Perform a Type II ANOVA (recommended for unbalanced/unequal group sizes)
anova_table = sm.stats.anova_lm(model, typ=2)

# Display the results
print(anova_table)

                        sum_sq      df          F        PR(>F)
C(retriever)          0.015292     1.0   0.115013  7.345245e-01
C(distance_metric)    0.030584     2.0   0.115013  7.345245e-01
C(model)             61.655809     8.0  57.965693  4.912581e-90
Residual            556.161396  4183.0        NaN           NaN


/Users/maritvandenhelder/miniconda3/envs/thesis/lib/python3.14/site-packages/statsmodels/base/model.py:1894: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 2, but rank is 1
  warnings.warn('covariance of constraints does not have full '
